# Januário Leads — Fase 8: complementar contatos via Geoapify Free

**Objetivo:** tentar localizar nome, endereço, telefone, e-mail, site e Instagram em lugares do Geoapify que correspondam às empresas da amostra Overture. Este notebook NÃO faz chamadas ao Google Places, pois copiar em massa seus dados para um banco de leads contraria as restrições de armazenamento e não atende ao requisito de custo zero.

### Proteções
- **Somente HOMOLOGAÇÃO** (`prlrxjothrslhswpkoza`). A produção e a busca do SaaS ficam intocadas.
- Conta **Geoapify Free sem cartão** obrigatória. Faça a confirmação manual no painel Geoapify antes de executar. Plano Free: **até 3.000 créditos diários**; este script restringe **250 créditos/dia, 25 por execução**, reservados atomicamente antes das consultas. Não use a mesma chave em outra automação que possa consumir o restante da cota sem controle.
- **Modo inicial = `no_contact`**: procura primeiro entre as empresas que não têm telefone, e-mail nem site. Outros modos: `any_missing`, `all` (para eventualmente tentar todas as 289.744).
- Consulta por **nome exato + ponto geográfico + categorias amplas**, máximo de 20 resultados por pedido, raio de 300 metros. Só apresenta uma correspondência com similaridade de nome >= 0,82 e distância <= 180 m.
- Nenhum dado Overture é sobrescrito. Os candidatos ficam separados em `overture_geoapify_enrichment` com `reviewed=false`; não constituem contatos confirmados.
- Interrompe em erro 402/429 de provedor, em falta de créditos e acima de **210 MiB** de banco; não tenta contornar cotas. Não usa chave Google nem faz consulta aos sites.
- **Dados do Geoapify**: guardar a origem e, se exibidos no futuro, mostrar [Geoapify](https://www.geoapify.com/) e [OpenStreetMap](https://www.openstreetmap.org/copyright) conforme seus termos.

### Instruções
1. Execute a **célula 1** para preparar as funções e a **célula 2** para testar a segurança de correspondência. Não fazem consultas externas.
2. Execute a **célula 3** e digite `GEOAPIFY_FREE_25`. As duas chaves serão digitadas em campos ocultos, sem salvamento no notebook.
3. A cada execução, serão pesquisadas até 25 empresas novas, sem repetir os IDs já processados. Para continuar, execute novamente apenas a célula 3. O limite diário de 250 é partilhado entre sessões.
4. Envie aqui apenas o trecho **RELATÓRIO FASE 8**, sem credenciais, para avaliar se compensa ampliar os lotes.

**Limitação:** o Geoapify também se baseia principalmente no OpenStreetMap; é possível que não acrescente dados a várias empresas. Mesmo com limite diário gratuito, uma tentativa individual para todas as empresas levaria muito tempo, e a política de conta Free precisa ser respeitada.

In [ ]:
# CÉLULA 1 — configurações e utilitários puros (NÃO acessa a internet)
import datetime as dt
import difflib
import getpass
import json
import math
import re
import time
import unicodedata
from urllib.parse import urlencode, urlparse
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

STAGING = "https://prlrxjothrslhswpkoza.supabase.co"
PROJECT_REF = "prlrxjothrslhswpkoza"
GEOAPIFY_ENDPOINT = "https://api.geoapify.com/v2/places"
MODE = "no_contact"        # Depois: "any_missing" ou "all"
MAX_REQUESTS_RUN = 25      # Máximo por execução, não aumente sem necessidade
MAX_REQUESTS_DAY = 250     # Limite GLOBAL no Supabase para este pipeline
STOP_DB_MIB = 210.0
RADIUS_METERS = 300
MATCH_MAX_DISTANCE_METERS = 180
MATCH_NAME_THRESHOLD = 0.82
CATEGORIES = "commercial,catering,healthcare,accommodation,office,service,leisure,sport,education,entertainment,tourism"
GEOAPIFY_LIMIT = 20
PAUSE_SECONDS = 1.1
assert MODE in ("no_contact","any_missing","all")
assert 1 <= MAX_REQUESTS_RUN <= 25
assert 1 <= MAX_REQUESTS_DAY <= 250
assert GEOAPIFY_LIMIT == 20 and RADIUS_METERS == 300

def normalized(value):
    value = unicodedata.normalize("NFKD",str(value or ""))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9]+"," ",value.lower())).strip()

def sim(a,b):
    x,y=normalized(a),normalized(b)
    if not x or not y: return 0.0
    return difflib.SequenceMatcher(None,x,y).ratio()

def haversine(a,b,c,d):
    x1,y1,x2,y2=map(math.radians,(a,b,c,d))
    delta_x,delta_y=x2-x1,y2-y1
    q=math.sin(delta_x/2)**2+math.cos(x1)*math.cos(x2)*math.sin(delta_y/2)**2
    return 2*6371000*math.asin(min(1,math.sqrt(q)))

def clean_phone(value):
    if not isinstance(value,str): return None
    raw=re.sub(r"[^0-9+]","",value.strip())[:28]
    digits=re.sub(r"\D","",raw)
    local=digits[2:] if digits.startswith("55") and len(digits)>=12 else digits
    return raw if re.fullmatch(r"[0-9]{10,11}",local) else None

def clean_email(value):
    if not isinstance(value,str):return None
    e=value.strip().split(";")[0].split(",")[0].lower()[:180]
    return e if re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}",e) else None

def clean_web(value):
    if not isinstance(value,str) or len(value)>400:return None
    v=value.strip()
    if v.startswith("www."):v="https://"+v
    try:
        u=urlparse(v)
        host=(u.hostname or "").lower()
        if u.scheme not in ("https","http") or not host or "." not in host or u.username or u.password:
            return None
        if host in ("localhost","example.com") or host.endswith((".local",".internal",".test")):
            return None
        return v
    except ValueError:
        return None

def clean_instagram(value):
    if not isinstance(value,str):return None
    s=value.strip()
    if s.startswith("@"):s="https://www.instagram.com/"+s[1:]
    elif s.lower().startswith("instagram.com/"):s="https://www."+s
    elif s.lower().startswith("www.instagram.com/"):s="https://"+s
    try:
        u=urlparse(s)
        if u.scheme not in ("http","https") or (u.hostname or "").lower() not in ("instagram.com","www.instagram.com"):
            return None
        parts=[x for x in u.path.split("/") if x]
        if len(parts)!=1 or parts[0].lower() in ("p","reel","stories","explore","accounts"):
            return None
        if not re.fullmatch(r"[a-zA-Z0-9_.]{2,30}",parts[0]):
            return None
        return "https://www.instagram.com/"+parts[0]+"/"
    except ValueError:
        return None

def _first(values,fn):
    for value in values:
        out=fn(value)
        if out:return out
    return None

def candidate_fields(feature):
    p=feature.get("properties",{}) if isinstance(feature,dict) else {}
    if not isinstance(p,dict):return {}
    contact=p.get("contact") if isinstance(p.get("contact"),dict) else {}
    ds=p.get("datasource") if isinstance(p.get("datasource"),dict) else {}
    raw=ds.get("raw") if isinstance(ds.get("raw"),dict) else {}
    coords=(feature.get("geometry") or {}).get("coordinates",[])
    if not isinstance(coords,list) or len(coords)<2 or not all(isinstance(i,(int,float)) for i in coords[:2]):return {}
    email=_first((contact.get("email"),p.get("email"),raw.get("contact:email"),raw.get("email")),clean_email)
    website=_first((p.get("website"),contact.get("website"),raw.get("contact:website"),raw.get("website")),clean_web)
    phone=_first((contact.get("phone"),p.get("phone"),raw.get("contact:phone"),raw.get("phone")),clean_phone)
    instagram=_first((contact.get("instagram"),p.get("instagram"),raw.get("contact:instagram"),raw.get("instagram")),clean_instagram)
    address=p.get("formatted") or p.get("address_line1")
    category=p.get("categories")
    return {"name":p.get("name"),"lon":float(coords[0]),"lat":float(coords[1]),
            "place_id":p.get("place_id"),"phone":phone,"email":email,"website":website,
            "instagram":instagram,"address":str(address)[:400] if isinstance(address,str) else None,
            "categories":[str(x)[:100] for x in category[:10]] if isinstance(category,list) else []}

def best_match(overture,features):
    matches=[]
    for f in features:
        x=candidate_fields(f)
        if not x.get("name") or not x.get("place_id"):continue
        d=haversine(float(overture["latitude"]),float(overture["longitude"]),x["lat"],x["lon"])
        name_score=sim(overture["name"],x["name"])
        if d>MATCH_MAX_DISTANCE_METERS or name_score<MATCH_NAME_THRESHOLD:
            continue
        x["distance_m"]=int(round(d))
        x["name_similarity"]=round(name_score,4)
        matches.append(x)
    return max(matches,key=lambda x:(x["name_similarity"],-x["distance_m"]),default=None)

def rest(secret,path,method="GET",payload=None,prefer=None):
    assert path.startswith("/rest/v1/") and not path.startswith("//")
    headers={"apikey":secret,"Accept":"application/json","Content-Type":"application/json"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    if prefer:headers["Prefer"]=prefer
    raw=json.dumps(payload,ensure_ascii=False).encode() if payload is not None else None
    req=Request(STAGING+path,data=raw,headers=headers,method=method)
    with urlopen(req,timeout=25) as response:
        body=response.read(1_000_000)
        return json.loads(body.decode("utf-8")) if body else None

def geoapify_places(key,target):
    query=urlencode({
      "apiKey":key,"categories":CATEGORIES,"name":target["name"][:150],
      "filter":"circle:{:.6f},{:.6f},{}".format(target["longitude"],target["latitude"],RADIUS_METERS),
      "limit":str(GEOAPIFY_LIMIT),"lang":"pt",
    })
    request=Request(GEOAPIFY_ENDPOINT+"?"+query,headers={"Accept":"application/json"})
    with urlopen(request,timeout=15) as response:
        payload=response.read(1_000_000)
        document=json.loads(payload.decode())
        if not isinstance(document,dict) or not isinstance(document.get("features"),list):
            raise ValueError("resposta Geoapify inesperada")
        return document["features"]

def quota_credit(secret):
    out=rest(secret,"/rest/v1/rpc/claim_overture_geoapify_free_credit",
             "POST",{"p_daily_limit":MAX_REQUESTS_DAY})
    return out is True

def storage_mib(secret):
    value=rest(secret,"/rest/v1/rpc/overture_geoapify_storage_mib","POST",{})
    return float(value)

def next_targets(secret,count):
    return rest(secret,"/rest/v1/rpc/next_overture_geoapify_targets",
                "POST",{"p_mode":MODE,"p_limit":count})

def save_suggestion(secret,row):
    rest(secret,"/rest/v1/overture_geoapify_enrichment?on_conflict=overture_id",
         "POST",row,prefer="resolution=merge-duplicates,return=minimal")

def status_counts(secret):
    for path in (
        "/rest/v1/overture_geoapify_enrichment?select=status",
        "/rest/v1/overture_geoapify_free_credit_usage?select=utc_day,credits_reserved&order=utc_day.desc&limit=2"
    ):
        # Avoid scanning the table of all companies just to build the summary.
        if "select=status" in path: continue
        return rest(secret,path)
print("Funções prontas. Nenhuma chave ou consulta externa utilizada.")


In [ ]:
# CÉLULA 2 — testes locais, sem consumir créditos
assert normalized("São João Barber")=="sao joao barber"
assert sim("Barbearia do Carlos","Barbearia do Carlos") == 1.0
assert sim("Barbearia do Carlos","Restaurante Pedro") < MATCH_NAME_THRESHOLD
assert round(haversine(-21.25,-45.0,-21.25,-45.0))==0
assert clean_email("contato@exemplo.com")=="contato@exemplo.com"
assert clean_instagram("https://www.instagram.com/barbearia_carlos/")=="https://www.instagram.com/barbearia_carlos/"
assert clean_instagram("https://instagram.com/p/ABCDE")==None
assert candidate_fields({"properties":{},"geometry":{"coordinates":[-45.,-21.]}}).get("place_id") is None
sample={"id":"teste","name":"Barbearia do Carlos","latitude":-21.25,"longitude":-45.0,
        "phone":None,"email":None,"website":None,"instagram":None}
feature={"properties":{"name":"Barbearia do Carlos","place_id":"geo-test",
        "contact":{"email":"contato@exemplo.com","phone":"+5535999999999",
        "website":"https://www.barber.com.br","instagram":"@barber_carlos"}},
        "geometry":{"coordinates":[-45.0,-21.25]}}
assert best_match(sample,[feature])["place_id"]=="geo-test"
wrong={"properties":{"name":"Padaria Geral","place_id":"geo-wrong"},
       "geometry":{"coordinates":[-45.0,-21.25]}}
assert best_match(sample,[wrong]) is None
far={"properties":{"name":"Barbearia do Carlos","place_id":"geo-far"},
       "geometry":{"coordinates":[-45.05,-21.25]}}
assert best_match(sample,[far]) is None
print("TESTES APROVADOS: validação de nome, posição, e-mail, Instagram e proteção contra falsos positivos.")


In [ ]:
# CÉLULA 3 — execução MANUAL: no máximo 25 pedidos em conta Geoapify Free
def run_geoapify_free():
    print("FASE 8 — teste controlado de enriquecimento (não inclui Google Places).")
    print("Modo:",MODE,"| teto por execução:",MAX_REQUESTS_RUN,
          "| reserva diária deste pipeline:",MAX_REQUESTS_DAY,"créditos.")
    print("Se a conta Geoapify tiver cobrança ativa, NÃO execute; o script não controla seu faturamento.")
    confirm=input("Confirma Geoapify FREE sem cartão e quer executar? Digite GEOAPIFY_FREE_25: ").strip()
    if confirm != "GEOAPIFY_FREE_25":
        print("CANCELADO. Nenhuma consulta externa feita.")
        return
    key=getpass.getpass("Chave SECRETA do Supabase HOMOLOGAÇÃO (campo oculto): ").strip()
    api=getpass.getpass("Chave do projeto Geoapify no plano FREE (campo oculto): ").strip()
    if not (key.startswith("sb_secret_") or key.startswith("eyJ")) or not api or len(api)>256:
        print("Credencial inválida ou ausente. Execução encerrada.")
        return
    started=dt.datetime.now(dt.timezone.utc)
    calls=0; matched=0; no_match=0; extra_fields={"phone":0,"email":0,"website":0,"instagram":0,"address":0}
    try:
        size=storage_mib(key)
        print("Banco de HOMOLOGAÇÃO:",round(size,2),"MiB")
        if size>=STOP_DB_MIB:
            print("BLOQUEADO: o banco alcançou o limite preventivo.")
            return
        targets=next_targets(key,MAX_REQUESTS_RUN)
        if not isinstance(targets,list):
            print("Alvos indisponíveis no Supabase.")
            return
        print("Empresas elegíveis neste lote:",len(targets))
        for item in targets:
            if calls>=MAX_REQUESTS_RUN:break
            if storage_mib(key)>=STOP_DB_MIB:
                print("PAUSA: limite de armazenamento preventivo.")
                break
            if not isinstance(item,dict) or not item.get("id") or not item.get("name"):
                continue
            # Spending is reserved atomically in the STAGING database BEFORE each call.
            if not quota_credit(key):
                print("PAUSA: limite diário de 250 créditos atingido.")
                break
            calls+=1
            try:
                features=geoapify_places(api,item)
            except HTTPError as error:
                print("PAUSA: Geoapify respondeu HTTP",error.code,
                      "na consulta",calls,"(sem imprimir URLs, chaves ou respostas).")
                break
            except (URLError,TimeoutError,ValueError) as error:
                print("PAUSA: falha de rede/resposta",type(error).__name__,
                      "na consulta",calls,". Nenhum resultado presumido.")
                break
            candidate=best_match(item,features)
            record={"overture_id":item["id"],"status":"no_match" if candidate is None else "matched",
                    "checked_at":dt.datetime.now(dt.timezone.utc).isoformat(),"reviewed":False}
            if candidate:
                def missing(field):return not str(item.get(field) or "").strip()
                record.update({
                  "candidate_name":str(candidate["name"])[:180],
                  "candidate_address":candidate["address"] if missing("address") else None,
                  "candidate_phone":candidate["phone"] if missing("phone") else None,
                  "candidate_email":candidate["email"] if missing("email") else None,
                  "candidate_website":candidate["website"] if missing("website") else None,
                  "candidate_instagram":candidate["instagram"] if missing("instagram") else None,
                  "geoapify_place_id":str(candidate["place_id"])[:256],
                  "geoapify_categories":candidate["categories"],
                  "matched_distance_m":candidate["distance_m"],
                  "matched_name_similarity":candidate["name_similarity"],
                  "note":"Sem validação de titularidade; revisar antes de usar contatos.",
                })
            try:
                save_suggestion(key,record)
            except HTTPError as error:
                print("PAUSA: não foi possível salvar a sugestão no Supabase, HTTP",error.code)
                break
            if candidate:
                matched+=1
                for field in extra_fields:
                    if record.get("candidate_"+field):extra_fields[field]+=1
                print("CANDIDATO:",item.get("name","")[:55],item.get("city",""),
                      item.get("state",""),"| distância",record["matched_distance_m"],"m",
                      "| novos campos",sum(bool(record.get("candidate_"+f)) for f in extra_fields))
            else:
                no_match+=1
                print("SEM CORRESPONDÊNCIA:",item.get("name","")[:55],
                      item.get("city",""),item.get("state",""))
            time.sleep(PAUSE_SECONDS)
        print("\n"+"="*55)
        print("RELATÓRIO FASE 8 — Geoapify Free")
        print("Consultas reservadas nesta execução:",calls)
        print("Correspondências candidatas:",matched)
        print("Consultas sem correspondência:",no_match)
        print("Novos telefones sugeridos:",extra_fields["phone"])
        print("Novos e-mails sugeridos:",extra_fields["email"])
        print("Novos sites sugeridos:",extra_fields["website"])
        print("Novos Instagrams sugeridos:",extra_fields["instagram"])
        print("Novos endereços sugeridos:",extra_fields["address"])
        print("Banco após execução:",round(storage_mib(key),2),"MiB")
        print("Google Places: ZERO requisições.")
        print("Contatos Overture originais: intactos; sugestão exige revisão.")
        print("Se desejar continuar, execute novamente esta célula; respeite o limite diário.")
        print("="*55)
    except HTTPError as error:
        print("Erro HTTP no banco de homologação:",error.code,"— nenhuma chave exibida.")
    finally:
        key=""
        api=""
        print("Credenciais descartadas desta execução. Desconecte o runtime ao terminar.")

run_geoapify_free()


## Depois do primeiro lote
Envie apenas o **RELATÓRIO FASE 8**. O objetivo inicial é medir a proporção de *novos contatos úteis* antes de gastar muitas requisições da cota gratuita.

Se as correspondências forem fracas, o melhor será mudar o método para consultas por categoria/cidade, capazes de avaliar vários estabelecimentos por consulta, sempre dentro dos créditos disponíveis.

**Não execute o notebook com uma conta Geoapify paga. Não coloque chaves em células. Não migre sugestões automaticamente para produção.**